# ExomSystem — Consultas SQL en Jupyter
Usamos `jupysql` para escribir SQL puro en celdas con `%%sql`.

> **Requisito:** `docker-compose up` corriendo antes de ejecutar.

## 0. Conexión a PostgreSQL

In [1]:
import sys
import os
from sqlalchemy import create_engine

sys.path.insert(0, "/app")

%load_ext sql

DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "civictech_db")
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASS = os.getenv("DB_PASSWORD", "")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

%sql engine

print(f"Conectado a {DB_NAME} en {DB_HOST}:{DB_PORT}")

There's a new jupysql version available (0.11.1), you're running 0.10.10. To upgrade: pip install jupysql --upgrade
Deploy Streamlit apps for free on Ploomber Cloud! Learn more: https://ploomber.io/s/signup
Conectado a civictech_db en db:5432


## 1. Verificar que PostGIS está activo

In [2]:
%%sql
SELECT PostGIS_Full_Version();

Running query in 'postgresql+psycopg2://postgres:***@db:5432/civictech_db'

1 rows affected.

postgis_full_version
"POSTGIS=""3.4.3 e365945"" [EXTENSION] PGSQL=""160"" GEOS=""3.9.0-CAPI-1.16.2"" PROJ=""7.2.1 NETWORK_ENABLED=OFF URL_ENDPOINT=https://cdn.proj.org USER_WRITABLE_DIRECTORY=/var/lib/postgresql/.local/share/proj DATABASE_PATH=/usr/share/proj/proj.db"" LIBXML=""2.9.10"" LIBJSON=""0.15"" LIBPROTOBUF=""1.3.3"" WAGYU=""0.5.0 (Internal)"" TOPOLOGY"


## 2. Ver las tablas del schema

In [3]:
%%sql
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;

Running query in 'postgresql+psycopg2://postgres:***@db:5432/civictech_db'

6 rows affected.

table_name
evidencia
geography_columns
geometry_columns
reporte
spatial_ref_sys
usuario


## 3. Consultas sobre usuarios
Tabla: `usuario` — columnas relevantes: `id_usuario`, `nombre_apellido`, `dni`, `email`, `reputacion`

In [4]:
%%sql
-- Todos los usuarios ordenados por reputación
-- BUGFIX: tabla 'usuario' (no 'usuarios'), columnas 'nombre_apellido' y 'reputacion'
SELECT id_usuario, nombre_apellido, email, reputacion
FROM usuario
ORDER BY reputacion DESC;

Running query in 'postgresql+psycopg2://postgres:***@db:5432/civictech_db'

4 rows affected.

id_usuario,nombre_apellido,email,reputacion
1,Mariano Ormeño,mariano.ormeno@email.com,100
2,Lucía Fanchil,lucia.f@email.com,98
4,Ana Pérez,ana@civictech.com,95
3,Carlos Castro,carlos.c@email.com,50


## 4. Consultas sobre reportes
Tabla: `reporte` — columnas relevantes: `id_reporte`, `id_usuario`, `patente_vehiculo`, `fechahora_dispositivo`, `fechahora_server`, `ubicacion`, `estado`, `descripcion`

In [5]:
%%sql
-- Todos los reportes con nombre de usuario
-- BUGFIX: tabla 'reporte' y 'usuario' (no 'reportes'/'usuarios')
--         columnas 'nombre_apellido', 'estado', 'fechahora_server'
--         eliminado JOIN a tipos_infraccion (tabla no existe en el schema)
SELECT
    r.id_reporte,
    u.nombre_apellido,
    r.patente_vehiculo,
    r.estado,
    r.descripcion,
    r.fechahora_server
FROM reporte r
JOIN usuario u ON u.id_usuario = r.id_usuario
ORDER BY r.fechahora_server DESC;

Running query in 'postgresql+psycopg2://postgres:***@db:5432/civictech_db'

4 rows affected.

id_reporte,nombre_apellido,patente_vehiculo,estado,descripcion,fechahora_server
4,Ana Pérez,AB123CD,Validada,Prueba desde Jupyter,2026-05-16 04:57:07.561417
1,Mariano Ormeño,AF123AA,Pendiente,Vehículo estacionado obstruyendo la rampa de accesibilidad en la esquina de la plaza principal.,2026-05-16 04:56:47.764131
2,Lucía Fanchil,AE987BB,Validada,"Auto en doble fila frente al ingreso escolar, interrumpiendo el tránsito de colectivos.",2026-05-16 04:56:47.764131
3,Carlos Castro,POV654,Rechazada,Estacionado en lugar no permitido.,2026-05-16 04:56:47.764131


In [6]:
%%sql
-- Solo los reportes pendientes
-- BUGFIX: tabla 'reporte', columnas 'estado' y 'fechahora_server'
SELECT id_reporte, patente_vehiculo, fechahora_server
FROM reporte
WHERE estado = 'Pendiente'
ORDER BY fechahora_server DESC;

Running query in 'postgresql+psycopg2://postgres:***@db:5432/civictech_db'

1 rows affected.

id_reporte,patente_vehiculo,fechahora_server
1,AF123AA,2026-05-16 04:56:47.764131


In [7]:
%%sql
-- Cantidad de reportes agrupados por estado
-- BUGFIX: reemplaza la query con tipos_infraccion (tabla inexistente)
SELECT
    estado,
    COUNT(*) AS total
FROM reporte
GROUP BY estado
ORDER BY total DESC;

Running query in 'postgresql+psycopg2://postgres:***@db:5432/civictech_db'

3 rows affected.

estado,total
Validada,2
Rechazada,1
Pendiente,1


## 5. Consultas espaciales PostGIS
La columna geográfica se llama `ubicacion` (no `coordenadas_gps`).

In [8]:
%%sql
-- Coordenadas GPS de cada reporte en formato legible (lon, lat)
-- BUGFIX: columna 'ubicacion' (no 'coordenadas_gps'), tabla 'reporte', columna 'estado'
SELECT
    id_reporte,
    patente_vehiculo,
    ST_X(ubicacion) AS longitud,
    ST_Y(ubicacion) AS latitud,
    estado
FROM reporte;

Running query in 'postgresql+psycopg2://postgres:***@db:5432/civictech_db'

4 rows affected.

id_reporte,patente_vehiculo,longitud,latitud,estado
1,AF123AA,-67.4965,-29.1611,Pendiente
2,AE987BB,-67.4892,-29.1664,Validada
3,POV654,-67.4915,-29.1548,Rechazada
4,AB123CD,-67.4965,-29.1611,Validada


In [9]:
%%sql
-- Reportes en un radio de 5 km desde el centro de Chilecito
-- Centro de Chilecito: lon=-67.4965, lat=-29.1611
-- BUGFIX: columna 'ubicacion', tabla 'reporte', columna 'estado'
SELECT
    id_reporte,
    patente_vehiculo,
    ST_X(ubicacion) AS longitud,
    ST_Y(ubicacion) AS latitud,
    ROUND(ST_Distance(
        ubicacion::geography,
        ST_SetSRID(ST_MakePoint(-67.4965, -29.1611), 4326)::geography
    )::numeric, 2) AS distancia_metros,
    estado
FROM reporte
WHERE ST_DWithin(
    ubicacion::geography,
    ST_SetSRID(ST_MakePoint(-67.4965, -29.1611), 4326)::geography,
    5000
)
ORDER BY distancia_metros;

Running query in 'postgresql+psycopg2://postgres:***@db:5432/civictech_db'

4 rows affected.

id_reporte,patente_vehiculo,longitud,latitud,distancia_metros,estado
1,AF123AA,-67.4965,-29.1611,0.00,Pendiente
4,AB123CD,-67.4965,-29.1611,0.00,Validada
3,POV654,-67.4915,-29.1548,851.02,Rechazada
2,AE987BB,-67.4892,-29.1664,921.65,Validada


## 6. Consultas sobre evidencias
Tabla: `evidencia` — columnas: `id_evidencia`, `id_infraccion` (FK a `reporte.id_reporte`), `url_foto`, `url_archivo_s3`, `hash_seguridad_sha`

In [10]:
%%sql
-- Evidencias con su reporte y patente asociada
-- BUGFIX: tabla 'evidencia' (no 'evidencias'), 'reporte' (no 'reportes')
--         columna 'id_infraccion' (no 'id_reporte'), 'hash_seguridad_sha' (no 'hash_seguridad_sha256')
SELECT
    e.id_evidencia,
    r.patente_vehiculo,
    e.url_archivo_s3,
    LEFT(e.hash_seguridad_sha, 16) || '...' AS hash_preview
FROM evidencia e
JOIN reporte r ON r.id_reporte = e.id_infraccion
ORDER BY e.id_evidencia;

Running query in 'postgresql+psycopg2://postgres:***@db:5432/civictech_db'

5 rows affected.

id_evidencia,patente_vehiculo,url_archivo_s3,hash_preview
1,AF123AA,https://s3.amazonaws.com/bucket-exomsystem/evidencias/2026/05/chilecito_001.jpg,e3b0c44298fc1c14...
2,AF123AA,https://s3.amazonaws.com/bucket-exomsystem/evidencias/2026/05/chilecito_001_alt.jpg,2cf24dba5fb0a30e...
3,AE987BB,https://s3.amazonaws.com/bucket-exomsystem/evidencias/2026/05/chilecito_002.jpg,751e22244d2b3e9f...
4,POV654,https://s3.amazonaws.com/bucket-exomsystem/evidencias/2026/05/chilecito_003.jpg,112e22244d2b3e9f...
5,AB123CD,https://mi-bucket.s3.amazonaws.com/fotos/AB123CD_001.jpg,735d6f25cfdb0c5c...


## 7. INSERT / UPDATE / DELETE desde SQL puro

In [11]:
%%sql
-- Insertar un usuario de prueba directo desde SQL
-- BUGFIX: tabla 'usuario', columnas 'nombre_apellido', 'reputacion'; se agrega 'contrasena' (NOT NULL)
INSERT INTO usuario (nombre_apellido, dni, email, contrasena, reputacion)
VALUES ('Carlos Test', '99999999', 'carlos@test.com', 'test_pass', 100)
ON CONFLICT (dni) DO NOTHING
RETURNING id_usuario, nombre_apellido;

Running query in 'postgresql+psycopg2://postgres:***@db:5432/civictech_db'

1 rows affected.

id_usuario,nombre_apellido
5,Carlos Test


In [12]:
%%sql
-- Cambiar estado de un reporte
-- BUGFIX: tabla 'reporte', columna 'estado' (no 'estado_resolucion')
--         valores válidos: 'Pendiente' | 'Validada' | 'Rechazada'
UPDATE reporte
SET estado = 'Validada'
WHERE id_reporte = 1
RETURNING id_reporte, estado;

Running query in 'postgresql+psycopg2://postgres:***@db:5432/civictech_db'

1 rows affected.

id_reporte,estado
1,Validada


In [13]:
%%sql
-- Eliminar el usuario de prueba
-- BUGFIX: tabla 'usuario', columna 'nombre_apellido'
DELETE FROM usuario
WHERE dni = '99999999'
RETURNING id_usuario, nombre_apellido;

Running query in 'postgresql+psycopg2://postgres:***@db:5432/civictech_db'

1 rows affected.

id_usuario,nombre_apellido
5,Carlos Test


## 8. Resultado como DataFrame de pandas

In [14]:
# pandas no está en libs.txt — lo instalamos si hace falta
try:
    import pandas
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "pandas", "-q"])

# BUGFIX: tabla 'reporte' (no 'reportes')
resultado = %sql SELECT * FROM reporte
df = resultado.DataFrame()
print(df.dtypes)
df.head()


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


Running query in 'postgresql+psycopg2://postgres:***@db:5432/civictech_db'

4 rows affected.

id_reporte                        int64
id_usuario                        int64
patente_vehiculo                    str
fechahora_dispositivo    datetime64[us]
fechahora_server         datetime64[us]
ubicacion                           str
estado                              str
hash_evidencia                      str
descripcion                         str
dtype: object


,id_reporte,id_usuario,patente_vehiculo,fechahora_dispositivo,fechahora_server,ubicacion,estado,hash_evidencia,descripcion
0,2,2,AE987BB,2026-05-16 02:56:47.764131,2026-05-16 04:56:47.764131,0101000020E6100000014D840D4FDF50C0ED0DBE30992A...,Validada,a591e1143c1a2d8e4115161c29e1f516a2b3c4d5e6f7a8...,"Auto en doble fila frente al ingreso escolar, ..."
1,3,3,POV654,2026-05-15 22:56:47.764131,2026-05-16 04:56:47.764131,0101000020E6100000FA7E6ABC74DF50C0BA6B09F9A027...,Rechazada,b682f2254d2b3e9f5226272d30f2a627b3c4d5e6f7a8b9...,Estacionado en lugar no permitido.
2,4,4,AB123CD,2026-05-16 04:57:07.561297,2026-05-16 04:57:07.561417,0101000020E6100000B29DEFA7C6DF50C0B7627FD93D29...,Validada,NaN,Prueba desde Jupyter
3,1,1,AF123AA,2026-05-16 04:11:47.764131,2026-05-16 04:56:47.764131,0101000020E6100000B29DEFA7C6DF50C0B7627FD93D29...,Validada,8f43428f53c153249d9c67ef1142bc3c4b14d23d8cbb81...,Vehículo estacionado obstruyendo la rampa de a...
